# Phase 6a - Export data for the dashboard

The dashboard runs on your **laptop** (Streamlit), not in Colab. This notebook collects every
small result file it needs into **one zip** in Drive:

| Part | Content |
|---|---|
| `eda/`, `sentiment/`, `fake_reviews/`, `recommendation/` | the CSV results of Phases 3-4 |
| `figures/` | the PNG charts |
| `sentiment_model.npz` | the trained sentiment model as plain numbers (vocabulary, IDF, weights) - so the dashboard can score typed reviews **without Spark** |
| `sentiment_check.csv` | Spark's predictions for 20 reviews, to prove the laptop version gives the same answers |
| `suspicious_examples.csv` | the 500 most suspicious reviews with their text and signals |
| `recommendations.csv`, `user_history.csv` | top-10 lists and what each user rated before |
| `rec_explanations.csv` | "recommended because you rated X" (explainability) |

**Run this after** the re-run of `04c_recommendation` (it needs the implicit ALS lists).
Takes about 5-10 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/amazon_reviews'
SAMPLE = '1pct'
OUT = '/content/dashboard_data'

import os
JAVA = '/usr/lib/jvm/java-17-openjdk-amd64'
if not os.path.exists(JAVA):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
os.environ['JAVA_HOME'] = JAVA
!pip -q install pyspark==3.5.3

!rm -rf {OUT} && mkdir -p {OUT}
for part in ['eda', 'sentiment', 'fake_reviews', 'recommendation', 'figures']:
    if os.path.exists(f'{PROJECT}/results/{part}'):
        !cp -r {PROJECT}/results/{part} {OUT}/
# the manual check sheets are for labelling, not for the dashboard
!rm -f {OUT}/fake_reviews/manual_check_*
!ls {OUT}

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.feature import StopWordsRemover
import numpy as np, pandas as pd

spark = (SparkSession.builder.appName('Phase6a-Export').master('local[*]')
         .config('spark.driver.memory', '8g').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
reviews = spark.read.parquet(f'{PROJECT}/parquet/reviews_clean_{SAMPLE}')
products = spark.read.parquet(f'{PROJECT}/parquet/products_clean_{SAMPLE}')

## 1. Sentiment model -> plain numbers

The saved Spark pipeline is: word pairs -> merge -> **CountVectorizer** (vocabulary) -> **IDF** ->
**Logistic Regression**. We export the vocabulary, the IDF weights, the LR weights and the stop-word
list (the same list Phase 2 used, keeping *not, no, nor, against, very, too*).

In [ ]:
model = PipelineModel.load(f'{PROJECT}/models/sentiment_lr_{SAMPLE}')
counter, idf_model, lr = model.stages[2], model.stages[3], model.stages[4]

KEEP_WORDS = {'not', 'no', 'nor', 'against', 'very', 'too'}
stop_words = [w for w in StopWordsRemover.loadDefaultStopWords('english') if w not in KEEP_WORDS]

np.savez_compressed(f'{OUT}/sentiment_model.npz',
                    vocab=np.array(counter.vocabulary),
                    idf=idf_model.idf.toArray(),
                    coef=lr.coefficientMatrix.toArray(),
                    intercept=lr.interceptVector.toArray(),
                    stop_words=np.array(stop_words))
print('vocabulary:', len(counter.vocabulary), '  classes:', lr.numClasses)

In [ ]:
# Spark's own predictions for 20 reviews - the dashboard's check uses these to prove it scores the same
from pyspark.ml.functions import vector_to_array
check = (model.transform(reviews.filter(F.size('tokens') > 2).orderBy(F.rand(7)).limit(20))
         .withColumn('p', vector_to_array('probability'))
         .select('title', 'text', F.col('p')[0].alias('p_negative'), F.col('p')[1].alias('p_neutral'),
                 F.col('p')[2].alias('p_positive'))
         .toPandas())
check.to_csv(f'{OUT}/sentiment_check.csv', index=False)
check[['title', 'p_negative', 'p_neutral', 'p_positive']].round(3).head()

## 2. Most suspicious reviews (with text)

In [ ]:
SIGNALS = ['near_duplicate', 'text_rating_mismatch', 'rating_deviation', 'burst',
           'outlier_reviewer', 'one_sided_reviewer', 'short_generic', 'unverified']
fake = spark.read.parquet(f'{PROJECT}/parquet/fake_scores_{SAMPLE}')
sentiment = spark.read.parquet(f'{PROJECT}/parquet/sentiment_scores_{SAMPLE}')
examples = (fake.filter('suspicious')
            .join(reviews.select('review_id', 'title', 'text', 'review_date', 'verified_purchase', 'word_count'), 'review_id')
            .join(products.select('parent_asin', 'product_title', 'average_rating'), 'parent_asin', 'left')
            .join(sentiment.select('review_id', 'p_negative', 'p_positive'), 'review_id', 'left')
            .orderBy(F.desc('suspicion_score'), F.rand(7))
            .limit(500)
            .select('review_id', 'user_id', 'product_title', 'rating', 'average_rating', 'review_date',
                    'verified_purchase', 'word_count', 'p_negative', 'p_positive',
                    'title', 'text', 'suspicion_score', 'signals_count', 'cluster', *SIGNALS)
            .toPandas())
examples.to_csv(f'{OUT}/suspicious_examples.csv', index=False)
print('suspicious examples:', len(examples))

## 3. Recommendations and each user's history (with product names)

In [ ]:
recs = spark.read.parquet(f'{PROJECT}/parquet/als_recommendations')
recs.toPandas().to_csv(f'{OUT}/recommendations.csv', index=False)

names = spark.read.parquet(f'{PROJECT}/parquet/product_names').select('parent_asin', 'title', 'main_category')
SCHEMA = 'user_id STRING, parent_asin STRING, rating DOUBLE, timestamp LONG'
history = (spark.read.csv([f'{PROJECT}/raw/5core/Electronics.{s}.csv.gz' for s in ['train', 'valid']],
                          header=True, schema=SCHEMA)
           .join(recs.select('user_id').distinct(), 'user_id')
           .join(names, 'parent_asin', 'left')
           .withColumn('date', F.to_date(F.from_unixtime(F.col('timestamp') / 1000)))
           .select('user_id', 'date', 'rating', 'parent_asin', 'title', 'main_category')
           .orderBy('user_id', 'date')
           .toPandas())
history.to_csv(f'{OUT}/user_history.csv', index=False)

# explainability: "recommended because you rated X" (made by 04c_recommendation section 8b)
expl_path = f'{PROJECT}/parquet/als_explanations'
if os.path.exists(expl_path):
    spark.read.parquet(expl_path).toPandas().to_csv(f'{OUT}/rec_explanations.csv', index=False)
    print('recommendation explanations exported')
else:
    print('no als_explanations yet - re-run 04c_recommendation.ipynb (section 8b)')
print('users with recommendations:', recs.select('user_id').distinct().count(), '  history rows:', len(history))

## 4. Zip it and save to Drive

In [ ]:
!cd /content && rm -f dashboard_data.zip && zip -qr dashboard_data.zip dashboard_data
!cp /content/dashboard_data.zip {PROJECT}/dashboard_data.zip
!ls -lh {PROJECT}/dashboard_data.zip

## Done - next steps on your laptop

1. In Google Drive open `amazon_reviews/`, right-click **`dashboard_data.zip`** -> **Download**.
2. Extract it into `D:\PROJECTS\BIG_DATA_project\dashboard\` so you get
   `D:\PROJECTS\BIG_DATA_project\dashboard\dashboard_data\...`
3. Follow `dashboard\README.md` to start the dashboard.